# `sync_gemini_observation_calendar` Demo

This notebook demonstrates the `sync_gemini_observation_calendar` management command,
which reads all `ObservationRecord(facility='GEM')` rows from the database and creates
or updates `CalendarEvent` rows idempotently.

## Scenarios covered

1. **Explicit window** — record with `windowDate`/`windowTime`/`windowDuration` present
   (GEM-WINDOW-01: primary happy path).
2. **Rap: derived window** — record with no explicit window; obs code maps to a `'Rap: ...'`
   settings entry → `[record.created, record.created + 24h]` (GEM-WINDOW-02).
3. **Std: derived window** — record with no explicit window; obs code maps to a `'Std: ...'`
   settings entry → `[record.created + 24h, record.created + 7d]` (GEM-WINDOW-02).
4. **ON_HOLD + idempotent re-run** — record with `ready='false'` shows `[ON_HOLD]` prefix;
   re-running the command produces no new events and leaves `CalendarEvent.modified` unchanged
   (GEM-STATUS-01, GEM-NOCHURN-01).

In [1]:
import os
import sys
from pathlib import Path

import django

repo_root_path = Path.cwd().resolve().parents[2]
assert (
    repo_root_path / 'manage.py'
).exists(), f'manage.py not found under {repo_root_path} — run this notebook from docs/notebooks/pre_executed/'
repo_root = str(repo_root_path)
src_root = str(repo_root_path / 'src')
for p in [repo_root, src_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')
django.setup()
print('Django setup complete')

Django setup complete


In [2]:
from django.conf import settings

# D-07: patch in a minimal FACILITIES['GEM']['programs'] block so instrument lookup
# and ToO-type detection run without real credentials.
# Program IDs follow real Gemini naming but end in '999' (D-05: fictitious, not confusable).
settings.FACILITIES.setdefault('GEM', {})
settings.FACILITIES['GEM']['programs'] = {
    'GS-2026A-T-999': {
        'MM': 'Std: GMOS-S MOS',  # Standard ToO template
        'QQ': 'Rap: GMOS-S MOS',  # Rapid ToO template (same instrument, different cadence)
    }
}
print('FACILITIES["GEM"]["programs"] patched:', settings.FACILITIES['GEM']['programs'])

FACILITIES["GEM"]["programs"] patched: {'GS-2026A-T-999': {'MM': 'Std: GMOS-S MOS', 'QQ': 'Rap: GMOS-S MOS'}}


In [3]:
from django.contrib.auth import get_user_model
from tom_calendar.models import CalendarEvent
from tom_observations.models import ObservationRecord
from tom_targets.models import Target
from tom_targets.tests.factories import NonSiderealTargetFactory

DEMO_TARGET_NAME = 'sync-gem-demo-target'
DEMO_PROG = 'GS-2026A-T-999'

# Create demo user and target (find-or-create to support re-runs).
demo_user, _ = get_user_model().objects.get_or_create(
    username='sync-gem-demo-user',
    defaults={'password': 'unused'},
)
demo_target = Target.objects.filter(name=DEMO_TARGET_NAME).first()
if demo_target is None:
    demo_target = NonSiderealTargetFactory.create(name=DEMO_TARGET_NAME)
print(f'Demo user pk={demo_user.pk}, target pk={demo_target.pk}')

User sync-gem-demo-user is not logged in. Cannot re-encrypt sensitive data. Clearing all encrypted fields instead.


Error clearing encrypted fields for model ESOProfile for user sync-gem-demo-user: Model instances passed to related filters must be saved.


No Profile found for sync-gem-demo-user. Creating Profile.


Target post save hook: sync-gem-demo-target created: True


Target post save hook: sync-gem-demo-target created: False


Demo user pk=2, target pk=1


## Create fixture ObservationRecords

Four records to exercise the four scenarios. All use `password: '[redacted]'` as a harmless
placeholder (real submissions would contain an encrypted key here; the command strips it
before any processing — D-04 / GEM-SECURE-01).

In [4]:
# Clean up any leftover demo records from prior runs.
ObservationRecord.objects.filter(user=demo_user, facility='GEM').delete()
CalendarEvent.objects.filter(url__startswith=f'GEM:{DEMO_PROG}/').delete()


def make_gem_record(obs_id: str, params: dict) -> ObservationRecord:
    return ObservationRecord.objects.create(
        observation_id=obs_id,
        target=demo_target,
        user=demo_user,
        facility='GEM',
        status='PENDING',
        parameters=params,
    )


# Scenario 1: explicit window (GEM-WINDOW-01)
rec_explicit = make_gem_record(
    '9001',
    {
        'prog': DEMO_PROG,
        'obsid': ['MM'],
        'ready': 'true',
        'password': '[redacted]',
        'windowDate': '2026-07-15',
        'windowTime': '02:30',
        'windowDuration': '4',
    },
)

# Scenario 2: Rap: derived window (GEM-WINDOW-02, QQ -> Rap:)
rec_rap = make_gem_record(
    '9002',
    {
        'prog': DEMO_PROG,
        'obsid': ['QQ'],
        'ready': 'true',
        'password': '[redacted]',
    },
)

# Scenario 3: Std: derived window (GEM-WINDOW-02, MM -> Std:)
rec_std = make_gem_record(
    '9003',
    {
        'prog': DEMO_PROG,
        'obsid': ['MM'],
        'ready': 'true',
        'password': '[redacted]',
    },
)

# Scenario 4: ON_HOLD (GEM-STATUS-01, ready='false')
rec_onhold = make_gem_record(
    '9004',
    {
        'prog': DEMO_PROG,
        'obsid': ['MM'],
        'ready': 'false',
        'password': '[redacted]',
    },
)

print('Created 4 ObservationRecords:', [rec_explicit.pk, rec_rap.pk, rec_std.pk, rec_onhold.pk])

Observation change state hook: sync-gem-demo-target @ GEM from None to PENDING


Observation change state hook: sync-gem-demo-target @ GEM from None to PENDING


Observation change state hook: sync-gem-demo-target @ GEM from None to PENDING


Observation change state hook: sync-gem-demo-target @ GEM from None to PENDING


Created 4 ObservationRecords: [1, 2, 3, 4]


## Run sync command (first pass)

In [5]:
import io
from datetime import timedelta
from datetime import timezone as dt_timezone

from django.core.management import call_command

stdout_buf = io.StringIO()
stderr_buf = io.StringIO()
call_command('sync_gemini_observation_calendar', stdout=stdout_buf, stderr=stderr_buf)

print('stdout:', stdout_buf.getvalue())
if stderr_buf.getvalue():
    print('stderr:', stderr_buf.getvalue())

stdout: Gemini South: created: 4, updated: 0, unchanged: 0, skipped: 0
Gemini North: created: 0, updated: 0, unchanged: 0, skipped: 0
Done.



## Inspect created CalendarEvents

In [6]:
from datetime import datetime

events = CalendarEvent.objects.filter(url__startswith=f'GEM:{DEMO_PROG}/').order_by('url')
print(f'Total CalendarEvents for {DEMO_PROG}: {events.count()}\n')
for ev in events:
    print(f'  url:        {ev.url}')
    print(f'  title:      {ev.title}')
    print(f'  telescope:  {ev.telescope}')
    print(f'  instrument: {ev.instrument}')
    print(f'  proposal:   {ev.proposal}')
    print(f'  start_time: {ev.start_time}')
    print(f'  end_time:   {ev.end_time}')
    print()

Total CalendarEvents for GS-2026A-T-999: 4

  url:        GEM:GS-2026A-T-999/9001
  title:      Gemini South GMOS-S MOS ToO
  telescope:  Gemini South
  instrument: GMOS-S MOS
  proposal:   GS-2026A-T-999
  start_time: 2026-07-15 02:30:00+00:00
  end_time:   2026-07-15 06:30:00+00:00

  url:        GEM:GS-2026A-T-999/9002
  title:      Gemini South GMOS-S MOS ToO
  telescope:  Gemini South
  instrument: GMOS-S MOS
  proposal:   GS-2026A-T-999
  start_time: 2026-06-27 03:57:41.295570+00:00
  end_time:   2026-06-28 03:57:41.295570+00:00

  url:        GEM:GS-2026A-T-999/9003
  title:      Gemini South GMOS-S MOS ToO
  telescope:  Gemini South
  instrument: GMOS-S MOS
  proposal:   GS-2026A-T-999
  start_time: 2026-06-28 03:57:41.298290+00:00
  end_time:   2026-07-04 03:57:41.298290+00:00

  url:        GEM:GS-2026A-T-999/9004
  title:      [ON_HOLD] Gemini South GMOS-S MOS ToO
  telescope:  Gemini South
  instrument: GMOS-S MOS
  proposal:   GS-2026A-T-999
  start_time: 2026-06-28 03:57:

## Scenario 1: Explicit window (GEM-WINDOW-01)

When `windowDate`/`windowTime`/`windowDuration` are present, the event's window is
derived directly from those values, not from the record's creation time.

In [7]:
ev_explicit = CalendarEvent.objects.get(url=f'GEM:{DEMO_PROG}/9001')
expected_start = datetime(2026, 7, 15, 2, 30, tzinfo=dt_timezone.utc)
expected_end = expected_start + timedelta(hours=4)

assert ev_explicit.start_time == expected_start, f'Expected {expected_start}, got {ev_explicit.start_time}'
assert ev_explicit.end_time == expected_end, f'Expected {expected_end}, got {ev_explicit.end_time}'
assert ev_explicit.instrument == 'GMOS-S MOS'
assert ev_explicit.telescope == 'Gemini South'
assert ev_explicit.proposal == DEMO_PROG
assert ev_explicit.title == 'Gemini South GMOS-S MOS ToO'
print('Scenario 1 (explicit window) OK')
print(f'  start_time = {ev_explicit.start_time}  (parsed from windowDate=2026-07-15, windowTime=02:30)')
print(f'  end_time   = {ev_explicit.end_time}  (start + 4h)')

Scenario 1 (explicit window) OK
  start_time = 2026-07-15 02:30:00+00:00  (parsed from windowDate=2026-07-15, windowTime=02:30)
  end_time   = 2026-07-15 06:30:00+00:00  (start + 4h)


## Scenario 2: Rapid ToO derived window (GEM-WINDOW-02, Rap:)

When no explicit window is present and the obs code maps to a `'Rap: ...'` settings entry,
the window is `[record.created, record.created + 24h]`.

In [8]:
ev_rap = CalendarEvent.objects.get(url=f'GEM:{DEMO_PROG}/9002')
rec_rap.refresh_from_db()  # ensure created timestamp is loaded

assert ev_rap.start_time == rec_rap.created, f'Expected {rec_rap.created}, got {ev_rap.start_time}'
assert ev_rap.end_time == rec_rap.created + timedelta(hours=24)
assert ev_rap.instrument == 'GMOS-S MOS'
assert ev_rap.title == 'Gemini South GMOS-S MOS ToO'
print('Scenario 2 (Rap: derived window) OK')
print(f'  start_time = {ev_rap.start_time}  (== record.created)')
print(f'  end_time   = {ev_rap.end_time}  (created + 24h)')

Scenario 2 (Rap: derived window) OK
  start_time = 2026-06-27 03:57:41.295570+00:00  (== record.created)
  end_time   = 2026-06-28 03:57:41.295570+00:00  (created + 24h)


## Scenario 3: Standard ToO derived window (GEM-WINDOW-02, Std:)

When no explicit window is present and the obs code maps to a `'Std: ...'` settings entry,
the window is `[record.created + 24h, record.created + 7d]`.

In [9]:
ev_std = CalendarEvent.objects.get(url=f'GEM:{DEMO_PROG}/9003')
rec_std.refresh_from_db()

assert ev_std.start_time == rec_std.created + timedelta(hours=24)
assert ev_std.end_time == rec_std.created + timedelta(days=7)
assert ev_std.instrument == 'GMOS-S MOS'
print('Scenario 3 (Std: derived window) OK')
print(f'  start_time = {ev_std.start_time}  (created + 24h)')
print(f'  end_time   = {ev_std.end_time}  (created + 7d)')

Scenario 3 (Std: derived window) OK
  start_time = 2026-06-28 03:57:41.298290+00:00  (created + 24h)
  end_time   = 2026-07-04 03:57:41.298290+00:00  (created + 7d)


## Scenario 4: ON_HOLD title prefix (GEM-STATUS-01)

When `ready='false'`, the event title is prefixed with `[ON_HOLD] `.

In [10]:
ev_onhold = CalendarEvent.objects.get(url=f'GEM:{DEMO_PROG}/9004')

assert ev_onhold.title.startswith('[ON_HOLD] '), f'Expected [ON_HOLD] prefix, got: {ev_onhold.title!r}'
print('Scenario 4 (ON_HOLD) OK')
print(f'  title = {ev_onhold.title!r}')

Scenario 4 (ON_HOLD) OK
  title = '[ON_HOLD] Gemini South GMOS-S MOS ToO'


## Scenario 4 (continued): Idempotent re-run (GEM-NOCHURN-01)

Running the command again on unchanged records must leave `CalendarEvent.modified`
untouched and report `unchanged: 4`.

In [11]:
# Snapshot modified timestamps before second run.
modified_before = {ev.pk: ev.modified for ev in events}

stdout_buf2 = io.StringIO()
call_command('sync_gemini_observation_calendar', stdout=stdout_buf2, stderr=io.StringIO())

print('stdout (second run):', stdout_buf2.getvalue())

# Verify no new events were created.
assert CalendarEvent.objects.filter(url__startswith=f'GEM:{DEMO_PROG}/').count() == 4

# Verify modified timestamps are unchanged.
for ev in CalendarEvent.objects.filter(url__startswith=f'GEM:{DEMO_PROG}/').order_by('url'):
    assert (
        ev.modified == modified_before[ev.pk]
    ), f'Event pk={ev.pk} modified changed from {modified_before[ev.pk]} to {ev.modified}'

assert 'unchanged: 4' in stdout_buf2.getvalue(), f'Expected "unchanged: 4" in: {stdout_buf2.getvalue()!r}'
print('Scenario 4 (idempotent re-run) OK — modified timestamps all unchanged')

stdout (second run): Gemini South: created: 0, updated: 0, unchanged: 4, skipped: 0
Gemini North: created: 0, updated: 0, unchanged: 0, skipped: 0
Done.

Scenario 4 (idempotent re-run) OK — modified timestamps all unchanged


## Teardown

Remove all demo fixtures so the notebook can be re-run cleanly.

In [12]:
deleted_events, _ = CalendarEvent.objects.filter(url__startswith=f'GEM:{DEMO_PROG}/').delete()
deleted_records, _ = ObservationRecord.objects.filter(user=demo_user, facility='GEM').delete()
print(f'Deleted {deleted_events} CalendarEvents and {deleted_records} ObservationRecords')

Deleted 4 CalendarEvents and 4 ObservationRecords
